In [13]:
import pandas as pd
from urllib.parse import urlparse
import re
import sys
!{sys.executable} -m pip install python-whois
import whois
from datetime import datetime

In [14]:
df=pd.read_csv(r'E:\UNMC Internship Document\Github Upload\data\dataset_live_links_v2.csv')

In [15]:
df.head()

,url,label
0,https://www.icecream.com,0
1,https://proud-lake-5717.kx7hcssq.workers.dev/,1
2,https://www.stmusic.at,0
3,http://icuwtzw.cloudaccess.host/,1
4,http://bcc56.freehostpro.com/,1


In [16]:
df['label'].value_counts()

label
1    441
0    363
Name: count, dtype: int64

In [17]:
def get_url_length(url):
    return len(url)

def get_has_https(url):
    return 1 if urlparse(url).scheme=="https" else 0

def get_num_subdomains(url):
    host=urlparse(url).netloc
    if host == "":
        return 0
    else:
        return host.count(".")-1
    

In [18]:
test=df.loc[1,'url']
test

'https://proud-lake-5717.kx7hcssq.workers.dev/'

In [19]:
print("Length:", get_url_length(test))
print(f"Has HTTPS: {get_has_https}")
print(f"Number of Subdomains: {get_num_subdomains(test)}")

Length: 45
Has HTTPS: <function get_has_https at 0x000001DD018C9BC0>
Number of Subdomains: 2


In [20]:
df['url_length']=df['url'].apply(get_url_length)
df['has_https']=df['url'].apply(get_has_https)
df['num_subdomains']=df['url'].apply(get_num_subdomains)

df.head()

,url,label,url_length,has_https,num_subdomains
0,https://www.icecream.com,0,24,1,1
1,https://proud-lake-5717.kx7hcssq.workers.dev/,1,45,1,2
2,https://www.stmusic.at,0,22,1,1
3,http://icuwtzw.cloudaccess.host/,1,32,0,1
4,http://bcc56.freehostpro.com/,1,29,0,1


In [21]:
def get_special_char_count(url):
    #count every character that is NOT a letter or digit
    return len(re.findall(r"[^a-zA-z0-9]",url))

def get_suspicious_keyword_count(url):
    #count how many scam-related urgency driven words appear in the url
    pattern=r"donate|urgent|help|verify|secure|account|login|gift|fund|claim"
    return len(re.findall(pattern, url.lower()))

In [22]:
test=df.loc[1,'url']
print("URL:",test)
print("Special chars:",get_special_char_count(test))
print("Suspicioius keywords:",get_suspicious_keyword_count(test))

URL: https://proud-lake-5717.kx7hcssq.workers.dev/
Special chars: 9
Suspicioius keywords: 0


In [23]:
df['Special Character Count']=df['url'].apply(get_special_char_count)
df['Suspicious Keyword Count']=df['url'].apply(get_suspicious_keyword_count)


df.head()

,url,label,url_length,has_https,num_subdomains,Special Character Count,Suspicious Keyword Count
0,https://www.icecream.com,0,24,1,1,5,0
1,https://proud-lake-5717.kx7hcssq.workers.dev/,1,45,1,2,9,0
2,https://www.stmusic.at,0,22,1,1,5,0
3,http://icuwtzw.cloudaccess.host/,1,32,0,1,6,0
4,http://bcc56.freehostpro.com/,1,29,0,1,6,0


In [24]:
res=whois.whois("spotify.com")

In [25]:
print(res)

{
  "domain_name": "SPOTIFY.COM",
  "registrar": "Abion AB",
  "registrar_url": [
    "http://www.abion.com",
    "www.abion.com"
  ],
  "reseller": null,
  "whois_server": "whois.abion.com",
  "referral_url": null,
  "updated_date": "2024-07-08 08:14:57+00:00",
  "creation_date": "2006-04-23 09:07:50+00:00",
  "expiration_date": "2030-04-23 09:07:50+00:00",
  "name_servers": [
    "DNS1.P07.NSONE.NET",
    "NS-CLOUD-A1.GOOGLEDOMAINS.COM",
    "NS-CLOUD-A2.GOOGLEDOMAINS.COM",
    "NS-CLOUD-A3.GOOGLEDOMAINS.COM",
    "NS-CLOUD-A4.GOOGLEDOMAINS.COM"
  ],
  "status": [
    "clientDeleteProhibited https://icann.org/epp#clientDeleteProhibited",
    "clientTransferProhibited https://icann.org/epp#clientTransferProhibited",
    "serverDeleteProhibited https://icann.org/epp#serverDeleteProhibited",
    "serverTransferProhibited https://icann.org/epp#serverTransferProhibited",
    "serverUpdateProhibited https://icann.org/epp#serverUpdateProhibited"
  ],
  "emails": "abuse@abion.com",
  "dnssec

In [26]:

def _clean_date(dt):
    """WHOIS dates can be a list, timezone-aware, or a string. Normalise them."""
    if dt is None:
        return None
    if isinstance(dt, list):
        dt = dt[0]
    if not isinstance(dt, datetime):
        return None
    if dt.tzinfo is not None:
        dt = dt.replace(tzinfo=None)
    return dt

def get_whois_data(url):
    """One lookup -> age, expiry, registrar."""
    try:
        info = whois.whois(url)
        created = _clean_date(info.creation_date)
        expiry  = _clean_date(info.expiration_date)
        age = (datetime.now() - created).days if created else -1
        exp = (expiry - datetime.now()).days  if expiry  else -1
        reg = str(info.registrar).lower() if info.registrar else "unknown"
        return age, exp, reg
    except Exception:
        return -1, -1, "unknown"


In [27]:
get_whois_data(df.loc[8,'url'])

Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed


(109, 255, 'spaceship, inc.')

In [28]:
import time

start = time.time()
results = []

for i, url in enumerate(df["url"], 1):
    results.append(get_whois_data(url))
    if i % 50 == 0:
        elapsed = time.time() - start
        print(f"  {i}/{len(df)} — {elapsed/60:.1f} min elapsed, "
              f"~{(elapsed/i)*(len(df)-i)/60:.1f} min left")

df["domain_age_days"]    = [r[0] for r in results]
df["domain_expiry_days"] = [r[1] for r in results]
df["registrar"]          = [r[2] for r in results]

print(f"\nDone in {(time.time()-start)/60:.1f} minutes")
print("Real ages found:", (df.domain_age_days != -1).sum(), "/", len(df))
print("Registrars found:", (df.registrar != "unknown").sum(), "/", len(df))

Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed


  50/804 — 1.6 min elapsed, ~23.4 min left


Error trying to connect to socket: closing socket - [WinError 10061] No connection could be made because the target machine actively refused it
Error trying to connect to socket: closing socket - [Errno 11002] getaddrinfo failed
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed


  100/804 — 2.9 min elapsed, ~20.2 min left


Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed


...

  700/804 — 34.9 min elapsed, ~5.2 min left


Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out


  750/804 — 38.1 min elapsed, ~2.7 min left


Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - [Errno 11001] getaddrinfo failed
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - timed out
Error trying to connect to socket: closing socket - timed out


  800/804 — 41.8 min elapsed, ~0.2 min left

Done in 41.9 minutes
Real ages found: 631 / 804
Registrars found: 620 / 804


In [29]:
df.to_csv('Feature Table.csv')

In [30]:
def unknown_check(registrar):
    if registrar=="unknown":
        return 1
    else:
        return 0
    
df['Unknown Registrar Status']=df['registrar'].apply(unknown_check)

In [31]:
df

,url,label,url_length,has_https,num_subdomains,Special Character Count,Suspicious Keyword Count,domain_age_days,domain_expiry_days,registrar,Unknown Registrar Status
0,https://www.icecream.com,0,24,1,1,5,0,11380,308,nom-iq ltd dba com laude,0
1,https://proud-lake-5717.kx7hcssq.workers.dev/,1,45,1,2,9,0,-1,-1,unknown,1
2,https://www.stmusic.at,0,22,1,1,5,0,-1,-1,world4you internet services gmbh ( https://nic...,0
3,http://icuwtzw.cloudaccess.host/,1,32,0,1,6,0,4223,524,"enom, llc",0
4,http://bcc56.freehostpro.com/,1,29,0,1,6,0,7927,107,"enom, inc.",0
...,...,...,...,...,...,...,...,...,...,...,...
799,https://www.prrn.mcgill.ca,0,26,1,2,6,0,9408,180,a.r.c. informatique inc.,0
800,https://www.shorthairmodels.com,0,31,1,1,5,0,3044,242,pdr ltd. d/b/a publicdomainregistry.com,0
801,https://www.amdea.org.uk,0,24,1,2,6,0,10375,216,elite limited t/a elite [tag = eliteuk],0
802,https://www.charitiesnys.com,0,28,1,1,5,0,6103,105,"godaddy.com, llc",0


In [58]:
info=whois.whois("http://bcc56.freehostpro.com/")
print(info)
registrar=info.registrar
registrar

{
  "domain_name": "FREEHOSTPRO.COM",
  "registrar": "ENOM, INC.",
  "registrar_url": [
    "http://www.enomdomains.com",
    "WWW.ENOMDOMAINS.COM"
  ],
  "reseller": null,
  "whois_server": "whois.enom.com",
  "referral_url": null,
  "updated_date": [
    "2025-09-03 10:39:15+00:00",
    "2026-07-15 22:18:58+00:00"
  ],
  "creation_date": [
    "2004-10-31 19:22:45+00:00",
    "2004-10-31 19:22:00+00:00"
  ],
  "expiration_date": [
    "2026-10-31 18:22:45+00:00",
    "2026-10-31 18:22:00+00:00"
  ],
  "name_servers": [
    "NS1.FREEHOSTPRO.COM",
    "NS2.FREEHOSTPRO.COM"
  ],
  "status": [
    "ok https://icann.org/epp#ok",
    "ok https://www.icann.org/epp#ok"
  ],
  "emails": "abuse@enom.com",
  "dnssec": "unsigned",
  "name": "REDACTED FOR PRIVACY",
  "org": "REDACTED FOR PRIVACY",
  "address": "REDACTED FOR PRIVACY",
  "city": "REDACTED FOR PRIVACY",
  "state": "SH",
  "registrant_postal_code": "REDACTED FOR PRIVACY",
  "country": "DE",
  "tech_name": "REDACTED FOR PRIVACY",
  "t

'ENOM, INC.'

In [62]:
import pandas as pd

lookup = pd.read_csv(
    r"E:\UNMC Internship Document\Github Upload\data\Phishing-RegistrarStats-Feb2026-Apr2026-cybercrimeinfocenter.csv"
)

# Clean lookup registrar names
lookup["Registrar Name"] = (
    lookup["Registrar Name"]
    .str.lower()
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(" incorporated", "", regex=False)
    .str.replace(" inc", "", regex=False)
    .str.replace(" llc", "", regex=False)
    .str.replace(" ltd", "", regex=False)
    .str.replace(" limited", "", regex=False)
    .str.replace(" corporation", "", regex=False)
    .str.replace(" corp", "", regex=False)
    .str.strip()
)

# Create lookup dictionary
rates = dict(zip(
    lookup["Registrar Name"],
    lookup["Phishing Domain Score"]
))

def get_registrar_phishing_score(reg):
    if pd.isna(reg):
        return -1

    reg = (
        reg.lower()
        .replace(",", "")
        .replace(".", "")
        .replace(" incorporated", "")
        .replace(" inc", "")
        .replace(" llc", "")
        .replace(" ltd", "")
        .replace(" limited", "")
        .replace(" corporation", "")
        .replace(" corp", "")
        .strip()
    )

    if reg == "unknown":
        return -2

    return rates.get(reg, -1)

# Clean registrar column
df["registrar"] = (
    df["registrar"]
    .str.lower()
    .str.replace(",", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(" incorporated", "", regex=False)
    .str.replace(" inc", "", regex=False)
    .str.replace(" llc", "", regex=False)
    .str.replace(" ltd", "", regex=False)
    .str.replace(" limited", "", regex=False)
    .str.replace(" corporation", "", regex=False)
    .str.replace(" corp", "", regex=False)
    .str.strip()
)

# Assign phishing scores
df["registrar_phishing_score"] = df["registrar"].apply(get_registrar_phishing_score)

In [63]:
df

,url,label,url_length,has_https,num_subdomains,Special Character Count,Suspicious Keyword Count,domain_age_days,domain_expiry_days,registrar,Unknown Registrar Status,registrar_phishing_score
0,https://www.icecream.com,0,24,1,1,5,0,11380,308,nom-iq dba com laude,0,1.00
1,https://proud-lake-5717.kx7hcssq.workers.dev/,1,45,1,2,9,0,-1,-1,unknown,1,-2.00
2,https://www.stmusic.at,0,22,1,1,5,0,-1,-1,world4you internet services gmbh ( https://nic...,0,-1.00
3,http://icuwtzw.cloudaccess.host/,1,32,0,1,6,0,4223,524,enom,0,3.84
4,http://bcc56.freehostpro.com/,1,29,0,1,6,0,7927,107,enom,0,3.84
...,...,...,...,...,...,...,...,...,...,...,...,...
799,https://www.prrn.mcgill.ca,0,26,1,2,6,0,9408,180,arc informatique,0,-1.00
800,https://www.shorthairmodels.com,0,31,1,1,5,0,3044,242,pdr d/b/a publicdomainregistrycom,0,29.32
801,https://www.amdea.org.uk,0,24,1,2,6,0,10375,216,elite t/a elite [tag = eliteuk],0,-1.00
802,https://www.charitiesnys.com,0,28,1,1,5,0,6103,105,godaddycom,0,2.48


In [61]:
df.to_csv('Feature Table 4.csv')

In [55]:
rates

{'network solutions llc': 1.32,
 'register.com - network solutions llc': 0.99,
 'enom llc': 3.84,
 'gmo internet group inc. d/b/a onamae.com': 14.46,
 'tucows domains inc.': 5.01,
 'scaleway sas': 4.51,
 'gandi sas': 3.84,
 'onlinenic inc.': 5.64,
 'ionos se': 2.25,
 'gkg.net inc.': 38.61,
 'ascio technologies inc. danmark - filial af ascio technologies inc. usa': 1.41,
 'csl computer service langenbach gmbh d/b/a joker.com': 1.96,
 'xin net technology corporation': 3.6,
 'cronon gmbh': 0.42,
 'godaddy.com llc': 2.48,
 'internetx gmbh': 19.8,
 'register spa': 2.76,
 'moniker online services llc': 2.01,
 'gabia inc.': 1.48,
 'key-systems gmbh': 7.6,
 'dnc holdings inc.': 3.02,
 'markmonitor inc.': 3.55,
 'csc corporate domains inc.': 0.5,
 'pdr ltd. d/b/a publicdomainregistry.com': 29.32,
 'arsys internet s.l. dba nicline.com': 0.8,
 'communigal communication ltd.': 4.03,
 'alibaba cloud computing (beijing) co. ltd.': 0.51,
 'dreamhost llc': 2.63,
 'ovh sas': 1.82,
 'wild west domains l

In [64]:
df

,url,label,url_length,has_https,num_subdomains,Special Character Count,Suspicious Keyword Count,domain_age_days,domain_expiry_days,registrar,Unknown Registrar Status,registrar_phishing_score
0,https://www.icecream.com,0,24,1,1,5,0,11380,308,nom-iq dba com laude,0,1.00
1,https://proud-lake-5717.kx7hcssq.workers.dev/,1,45,1,2,9,0,-1,-1,unknown,1,-2.00
2,https://www.stmusic.at,0,22,1,1,5,0,-1,-1,world4you internet services gmbh ( https://nic...,0,-1.00
3,http://icuwtzw.cloudaccess.host/,1,32,0,1,6,0,4223,524,enom,0,3.84
4,http://bcc56.freehostpro.com/,1,29,0,1,6,0,7927,107,enom,0,3.84
...,...,...,...,...,...,...,...,...,...,...,...,...
799,https://www.prrn.mcgill.ca,0,26,1,2,6,0,9408,180,arc informatique,0,-1.00
800,https://www.shorthairmodels.com,0,31,1,1,5,0,3044,242,pdr d/b/a publicdomainregistrycom,0,29.32
801,https://www.amdea.org.uk,0,24,1,2,6,0,10375,216,elite t/a elite [tag = eliteuk],0,-1.00
802,https://www.charitiesnys.com,0,28,1,1,5,0,6103,105,godaddycom,0,2.48


In [67]:
df["tld"] = df["url"].str.extract(r"\.([a-zA-Z]{2,})(?:/|$)")

In [68]:
df

,url,label,url_length,has_https,num_subdomains,Special Character Count,Suspicious Keyword Count,domain_age_days,domain_expiry_days,registrar,Unknown Registrar Status,registrar_phishing_score,tld
0,https://www.icecream.com,0,24,1,1,5,0,11380,308,nom-iq dba com laude,0,1.00,com
1,https://proud-lake-5717.kx7hcssq.workers.dev/,1,45,1,2,9,0,-1,-1,unknown,1,-2.00,dev
2,https://www.stmusic.at,0,22,1,1,5,0,-1,-1,world4you internet services gmbh ( https://nic...,0,-1.00,at
3,http://icuwtzw.cloudaccess.host/,1,32,0,1,6,0,4223,524,enom,0,3.84,host
4,http://bcc56.freehostpro.com/,1,29,0,1,6,0,7927,107,enom,0,3.84,com
...,...,...,...,...,...,...,...,...,...,...,...,...,...
799,https://www.prrn.mcgill.ca,0,26,1,2,6,0,9408,180,arc informatique,0,-1.00,ca
800,https://www.shorthairmodels.com,0,31,1,1,5,0,3044,242,pdr d/b/a publicdomainregistrycom,0,29.32,com
801,https://www.amdea.org.uk,0,24,1,2,6,0,10375,216,elite t/a elite [tag = eliteuk],0,-1.00,uk
802,https://www.charitiesnys.com,0,28,1,1,5,0,6103,105,godaddycom,0,2.48,com


In [69]:

TLD_Dataset = pd.read_csv(
    r'E:\UNMC Internship Document\Github Upload\data\Phishing-TLDstats-Feb2026-Apr2026.csv'
)

# Create lookup dictionary
tld_scores = dict(zip(
    TLD_Dataset["TLD"].str.lower(),
    TLD_Dataset["Phishing Domain Score"]
))

def get_TLD_Phishing_Score(tld):
    if pd.isna(tld):
        return -1

    tld = tld.lower()

    return tld_scores.get(tld, -1)

In [70]:
df['TLD Phishing Score']=df['tld'].apply(get_TLD_Phishing_Score)

In [71]:
df

,url,label,url_length,has_https,num_subdomains,Special Character Count,Suspicious Keyword Count,domain_age_days,domain_expiry_days,registrar,Unknown Registrar Status,registrar_phishing_score,tld,TLD Phishing Score
0,https://www.icecream.com,0,24,1,1,5,0,11380,308,nom-iq dba com laude,0,1.00,com,11.66
1,https://proud-lake-5717.kx7hcssq.workers.dev/,1,45,1,2,9,0,-1,-1,unknown,1,-2.00,dev,7.49
2,https://www.stmusic.at,0,22,1,1,5,0,-1,-1,world4you internet services gmbh ( https://nic...,0,-1.00,at,2.16
3,http://icuwtzw.cloudaccess.host/,1,32,0,1,6,0,4223,524,enom,0,3.84,host,-1.00
4,http://bcc56.freehostpro.com/,1,29,0,1,6,0,7927,107,enom,0,3.84,com,11.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799,https://www.prrn.mcgill.ca,0,26,1,2,6,0,9408,180,arc informatique,0,-1.00,ca,1.84
800,https://www.shorthairmodels.com,0,31,1,1,5,0,3044,242,pdr d/b/a publicdomainregistrycom,0,29.32,com,11.66
801,https://www.amdea.org.uk,0,24,1,2,6,0,10375,216,elite t/a elite [tag = eliteuk],0,-1.00,uk,1.42
802,https://www.charitiesnys.com,0,28,1,1,5,0,6103,105,godaddycom,0,2.48,com,11.66
